# Réplica paso 1: CodeBERT afinado (transfer learning)

Paper: Alturayeif & Hassine (2025), PeerJ Computer Science. Objetivo de esta parte: obtener un F2 cercano a **0.67** (Tabla 2 del paper) entrenando con los 1,523 fragmentos.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.

In [ ]:
!nvidia-smi

## 1. Traer el repositorio

In [1]:
!git clone https://github.com/priscilaGO/replica-data-leakage-detection.git
%cd replica-data-leakage-detection/AL4Code
!ls .. ../Dataset

Cloning into 'replica-data-leakage-detection'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 30 (delta 1), reused 27 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 196.16 KiB | 16.35 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/replica-data-leakage-detection/AL4Code
..:
AL4Code  GPT4Code		  notebooks   README.md
Dataset  LICENSE_AL4Code_MIT.txt  PATCHES.md  Results

../Dataset:
test_annotated.csv  train_annotated.csv


## 2. Instalar librerías

In [2]:
!pip install -q "transformers>=4.40,<5" "openai>=1.0" tenacity scikit-learn pandas nltk

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 3. Prueba de humo: una corrida (semilla 1)

Entrena CodeBERT 4 épocas con los 1,523 fragmentos y evalúa en los 381 del test. Tarda varios minutos. Mira al final las líneas `Round 0 testing ...`.

In [3]:
!python demo.py --seed 1 --tag passive_seed1

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
I0000 00:00:1794886786.582191    1075 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794886789.213762    1075 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
{'active_learning': False,
 'n_init_labeled': 20,
 'n_query': 20,
 'patient': 5,
 'seed': 1,
 'strategy_name': 'LeastConfidence',
 'tag': 'passive_seed1',
 'with_augmentation': False}

Loading BERT tokenizer...
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 187kB/s]
vocab.json: 899kB [00:00, 79.3MB/s]
merges.txt: 456kB [00:00, 98.2MB/s]
special_tokens_map.json: 100% 150/150 [00:00<00:00, 807kB/s]
config.json: 100% 498/498 [00:00<00:00, 4.90MB/s]
Token indice

## 4. Cinco corridas (semillas 1 a 5)

Ejecutar solo cuando la prueba de humo haya salido bien.

In [ ]:
for s in [2, 3, 4, 5]:
    !python demo.py --seed {s} --tag passive_seed{s}

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
I0000 00:00:1794887582.376408    4525 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794887585.021480    4525 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
{'active_learning': False,
 'n_init_labeled': 20,
 'n_query': 20,
 'patient': 5,
 'seed': 2,
 'strategy_name': 'LeastConfidence',
 'tag': 'passive_seed2',
 'with_augmentation': False}

Loading BERT tokenizer...
Token indices sequence length is longer than the specified maximum sequence length for this model (1133 > 512). Running this sequence through the model will result in indexing errors
Max sentence length:  8578
number of labeled pool: 20
number of unla

## 5. Resumen de resultados

In [ ]:
import pandas as pd, glob
rows = []
for f in sorted(glob.glob('../Results/LeastConfidence_passive_seed*.csv')):
    d = pd.read_csv(f).iloc[-1]
    rows.append({'corrida': f.split('_')[-1][:-4], 'precision': d.precision, 'recall': d.recall, 'f2': d.f2})
df = pd.DataFrame(rows)
display(df)
print('Promedio:'); print(df[['precision','recall','f2']].mean().round(2))
print('Desv. est. F2:', round(df.f2.std(), 2))
print('Paper (Tabla 2, CodeBERT): precision 0.83, recall 0.64, F2 0.67, desv. F2 0.03')

## 6. Guardar resultados en GitHub

Descarga la carpeta `Results` (clic derecho en el panel de archivos → Descargar) y súbela a tu repo.